In [1]:
import os
import sys
import time
import torch
import skimage
import sklearn.metrics
import wandb
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

import mnds
import mnmodel
import evaluation

# Bin the sizes

In [2]:
# Hyperparameters
CURRENT_PATH = os.getcwd()
DIRECTORY = CURRENT_PATH + '/dataset_v2'
OUTPUT_DIR = "/model_output/output/"
SCALE_FACTOR = 1.0
THRESHOLD = 0.5

In [3]:
filelist = os.listdir(DIRECTORY + OUTPUT_DIR)
IoU_files = [file for file in filelist if file.endswith('._IoUs.npy')]
pilot = [file for file in IoU_files if not file.startswith('C2-')]
screen = [file for file in IoU_files if file.startswith('C2-')]
print(len(pilot), len(screen))

10 8


In [5]:
def measures_at(threshold, IOU):
    matches = IOU > threshold
    
    true_positives = np.sum(matches, axis=1) == 1   # Correct objects
    false_positives = np.sum(matches, axis=0) == 0  # Extra objects
    false_negatives = np.sum(matches, axis=1) == 0  # Missed objects
    
    assert np.all(np.less_equal(true_positives, 1))
    assert np.all(np.less_equal(false_positives, 1))
    assert np.all(np.less_equal(false_negatives, 1))
    
    TP, FP, FN = np.sum(true_positives), np.sum(false_positives), np.sum(false_negatives)
    
    f1 = 2*TP / (2*TP + FP + FN + 1e-9)

    prec = TP / (TP + FP)

    rec = TP / (TP + FN)

    return f1, prec, rec, TP, FP, FN


In [10]:
BIN_SIZE = [[0,50], [50,100], [100, 150], [0,100], [150,np.inf]]

In [11]:
precision = []
recall = []
  
for bounds in BIN_SIZE:
    print(f'Bin Size: {bounds}')
    
    for i in range(len(pilot)):
        path = DIRECTORY + OUTPUT_DIR + pilot[i]
        matches = np.load(path)
        matches = matches[:,:-1]

        imid = pilot[i].split('.')[0]
        mn_gt = mnds.read_micronuclei_masks(DIRECTORY, imid, SCALE_FACTOR)
        gti = skimage.morphology.label(mn_gt)
        gt_area = []
        
        for j in range(matches.shape[0]):
            label_gt = j + 1
            area = np.sum(gti == label_gt)
            gt_area.append(area)
            
        matches = np.column_stack((matches, gt_area))
        matches= matches[matches[:, -1].argsort()]

        bin_matches = matches[(matches[:, -1] > bounds[0]) & (matches[:, -1] < bounds[1])]
        f1, prec, rec, TP, FP, FN = measures_at(0.1, bin_matches[:,:-1])
        precision.append(prec)
        recall.append(rec)
        
        # print(f'{imid} - Precision: {prec:.2f}, Recall: {rec:.2f}')
        
    print(f'(Average) Precision: {np.mean(precision):.2f}, Recall: {np.mean(recall):.2f}')

Bin Size: [0, 50]
(Average) Precision: 0.32, Recall: 0.77
Bin Size: [50, 100]
(Average) Precision: 0.24, Recall: 0.84
Bin Size: [100, 150]
(Average) Precision: 0.18, Recall: 0.87
Bin Size: [0, 100]
(Average) Precision: 0.26, Recall: 0.85
Bin Size: [150, inf]
(Average) Precision: 0.23, Recall: 0.87


In [12]:
precision = []
recall = []

for bounds in BIN_SIZE:
    print(f'Bin Size: {bounds}')
    
    for i in range(len(screen)):
        path = DIRECTORY + OUTPUT_DIR + screen[i]
        matches = np.load(path)
        matches = matches[:,:-1]

        imid = screen[i].split('.')[0]
        mn_gt = mnds.read_micronuclei_masks(DIRECTORY, imid, SCALE_FACTOR)
        gti = skimage.morphology.label(mn_gt)
        gt_area = []
        
        for j in range(matches.shape[0]):
            label_gt = j + 1
            area = np.sum(gti == label_gt)
            gt_area.append(area)
            
        matches = np.column_stack((matches, gt_area))
        matches= matches[matches[:, -1].argsort()]

        bin_matches = matches[(matches[:, -1] > bounds[0]) & (matches[:, -1] < bounds[1])]
        f1, prec, rec, TP, FP, FN = measures_at(0.1, bin_matches[:,:-1])
        precision.append(prec)
        recall.append(rec)
        
        # print(f'{imid} - Precision: {prec}, Recall: {rec}')
        
    print(f'(Average) Precision: {np.mean(precision):.2f}, Recall: {np.mean(recall):.2f}')

Bin Size: [0, 50]
(Average) Precision: 0.40, Recall: 0.79
Bin Size: [50, 100]
(Average) Precision: 0.30, Recall: 0.88
Bin Size: [100, 150]
(Average) Precision: 0.22, Recall: 0.91
Bin Size: [0, 100]
(Average) Precision: 0.32, Recall: 0.90
Bin Size: [150, inf]
(Average) Precision: 0.28, Recall: 0.91
